# Colour game probe — TransformerLens 3 (TransformerBridge)

Same experiment as before: run the four-turn colour game, read every CoT, then check
whether any colour is represented in the residual stream at each turn.

What moves to the library: model loading, module resolution, activation caching, and —
the important one — the layer-norm handling in the logit lens. `accumulated_resid(apply_ln=True)`
does what I was hand-rolling with `FINAL_NORM`, and it is tested against raw HF forward
parity rather than checked by me in a setup cell.

What stays hand-written: the J-lens (custom per-layer matrices, not a TL concept), the
colour token-id set, and the CoT parsing.

In [4]:
!pip install -q -U "transformer-lens>=3.0" accelerate pandas matplotlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 662.8 kB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 2.2 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 5.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 35.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 45.0 MB/s eta 0:00:0000:010:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 36.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 94.8 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed

## 1. Config

`ENABLE_COMPAT` is deliberately **False**. `enable_compatibility_mode()` applies
`fold_ln`, `center_writing_weights`, and `center_unembed` to match legacy
`HookedTransformer` defaults, which changes the numerical values in the residual stream.
The J-lens matrices were fitted against raw HuggingFace residuals, so folding and
centering would put the residuals out of the distribution `J` was estimated on. Leaving
it off keeps activations matching raw HF, which is what the lens expects.

That is a real decision, not a default — if you later use TL utilities that assume folded
LN, you have to revisit it.

In [2]:
MODEL_NAME  = "Qwen/Qwen3.5-4B"
LENS_REPO   = "camilablank/workspace-lenses"
LENS_SUBDIR = "qwen3.5-4b"

ENABLE_COMPAT  = False    # see markdown above - keep False while using the J-lens
MAX_NEW_TOKENS = 1000
TEMPERATURE    = 1.0
SEED           = 1234

TURNS = [
    "Think of a random color in your head. Just let me know once you've picked one "
    "but don't tell me what it is. I will guess it",
    "Is it purple?",
    "Is it blue?",
    "I give up. What color was it?",
]

import torch, re, math

## 2. Load via TransformerBridge

`boot_transformers` picks the architecture adapter, maps the HF module graph onto
generalized components, and registers hook points. Qwen3.5 is covered by the `qwen3_5`
adapters and the gated-delta-net family is included in the forward-parity benchmark
suite, so the module-path guessing from the raw-HF version is gone.

If this errors on the architecture, print `bridge.cfg` and check whether the checkpoint
resolves to the text-only or the multimodal adapter — Qwen3.5 has both.

In [6]:
from transformer_lens.model_bridge import TransformerBridge

bridge = TransformerBridge.boot_transformers(
    MODEL_NAME, device="cuda" if torch.cuda.is_available() else "cpu", dtype=torch.bfloat16
)
if ENABLE_COMPAT:
    bridge.enable_compatibility_mode()

tokenizer = bridge.tokenizer
DEV = bridge.cfg.device
print(f"n_layers={bridge.cfg.n_layers}  d_model={bridge.cfg.d_model}  d_vocab={bridge.cfg.d_vocab}")

# Unembedding matrix - attribute name differs slightly across versions.
W_U = getattr(bridge, "W_U", None)
if W_U is None:
    W_U = bridge.unembed.W_U
print("W_U:", tuple(W_U.shape))

config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/12.8M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

n_layers=32  d_model=2560  d_vocab=248320
W_U: (2560, 248320)


## 3. Parity check

One cell, and it replaces the whole pre-norm/post-norm investigation from the raw-HF
notebook. If reconstructing the final logits from the accumulated residual stack
reproduces what the model actually returned, then the residual indexing, the LN
application, and the unembed are all correct and every readout below inherits that.

If it fails, stop — nothing downstream is interpretable.

In [7]:
logits, cache = bridge.run_with_cache("The capital of France is Paris. The capital of Japan is")

resid, labels = cache.accumulated_resid(layer=-1, incl_mid=False, apply_ln=True, return_labels=True)
recon = (resid[-1] @ W_U)[0, -1].float()
ref   = logits[0, -1].float()

print("residual stack:", tuple(resid.shape), "| last label:", labels[-1])
print(f"max|reconstructed - actual| = {(recon - ref).abs().max().item():.4f}")
print("model top-5:", [tokenizer.decode([i]) for i in ref.topk(5).indices.tolist()])
print("recon top-5:", [tokenizer.decode([i]) for i in recon.topk(5).indices.tolist()])
assert (recon - ref).abs().max() < 0.5, "Residual stack does not reproduce the logits."
del cache, resid

residual stack: (33, 1, 12, 2560) | last label: final_post
max|reconstructed - actual| = 0.0000
model top-5: [' Tokyo', ' Kyoto', '...', ' ______', ' __']
recon top-5: [' Tokyo', ' Kyoto', '...', ' ______', ' __']


## 4. Chat + generation helpers

`bridge.generate` accepts a token tensor. Its exact signature has moved between versions,
so the call is wrapped in a fallback — this is the one place you may need to adjust.

(TL 3.x also added `generate(return_cache=True)`, which caches the full prompt+generation
in one pass. If your installed version has it, you can skip the separate `run_with_cache`
call in section 6.)

`last_content_index` scans from the **end**. Scanning forward lands on the `<|im_end|>`
closing the user turn, not on anything the model generated.

In [12]:
SPECIAL = set(tokenizer.all_special_ids)

def chat_tokens(messages):
    enc = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True, return_tensors="pt"
    )
    if hasattr(enc, "input_ids"):
        return enc.input_ids.to(DEV)
    return enc.to(DEV)

def generate_ids(tokens, seed=SEED):
    torch.manual_seed(seed)
    kw = dict(max_new_tokens=MAX_NEW_TOKENS, do_sample=True,
              temperature=TEMPERATURE, stop_at_eos=True, verbose=False)
    for drop in ([], ["verbose"], ["verbose", "stop_at_eos"]):
        try:
            return bridge.generate(tokens, **{k: v for k, v in kw.items() if k not in drop})
        except TypeError:
            continue
    raise RuntimeError("bridge.generate signature not matched - check your TL version")

def split_think(text):
    if "</think>" not in text:
        return text, ""
    cot, vis = text.split("</think>", 1)
    for t in tokenizer.all_special_tokens:
        vis = vis.replace(t, ""); cot = cot.replace(t, "")
    return cot.replace("<think>", "").strip(), vis.strip()

def last_content_index(ids):
    lst = ids[0].tolist()
    for i in range(len(lst) - 1, -1, -1):
        if lst[i] not in SPECIAL:
            return i
    return len(lst) - 1

def user(t):      return {"role": "user",      "content": t}
def assistant(t): return {"role": "assistant", "content": t}

## 5. Colour tokens

Token ids, not substrings — `"red" in tok` also fires on `" structured"` and
`" predicted"`. Leading-space and capitalised forms are separate ids; we keep every
single-token variant and take the best rank across them. Multi-token colours are dropped
and reported, since a single-position readout cannot see them.

In [13]:
COLOR_WORDS = ["red","orange","yellow","green","blue","purple","violet","indigo","pink",
               "black","white","brown","gray","grey","gold","silver","cyan","teal",
               "maroon","navy","beige","olive"]

COLOR_IDS, DROPPED = {}, []
for w in COLOR_WORDS:
    v = sorted({tokenizer.encode(s, add_special_tokens=False)[0]
                for f in (w, w.capitalize()) for s in (f, " " + f)
                if len(tokenizer.encode(s, add_special_tokens=False)) == 1})
    if v: COLOR_IDS[w] = v
    else: DROPPED.append(w)

COLORS   = sorted(COLOR_IDS)
COLOR_RE = re.compile(r"\b(" + "|".join(COLORS) + r")\b", re.I)
def named_colors(t): return sorted({m.group(1).lower() for m in COLOR_RE.finditer(t)})
print(len(COLORS), "colours | dropped (multi-token):", DROPPED)

20 colours | dropped (multi-token): ['indigo', 'maroon']


## 6. Run the four turns

In [19]:
# ---- cell 6: run the game ----

MAX_NEW_TOKENS = 4000   # yes it's slow. truncation is worse.

SETUP  = ("Think of a random color in your head. Just let me know once you've picked one "
          "but don't tell me what it is. I will guess it")
SCRIPT = ["Is it purple?", "Is it blue?", "I give up. What color was it?"]

def ask(history, text):
    history.append(user(text))
    toks = chat_tokens(history)
    out  = generate_ids(toks)
    full = tokenizer.decode(out[0], skip_special_tokens=False)
    cot, vis = split_think(full)
    return {"user": text, "cot": cot, "visible": vis, "ids": out,
            "prompt_len": toks.shape[1],
            "cot_colors": named_colors(cot), "vis_colors": named_colors(vis)}

history, turns = [], []
for text in [SETUP] + SCRIPT:
    t = ask(history, text)
    turns.append(t)
    t["i"] = len(turns) - 1
    print(f"turn {t['i']}: {text[:50]:52s} visible={repr(t['visible'][:80])}")
    if not t["visible"]:
        print("\n  STOP — think block never closed. Everything after this would be corrupted.")
        print(f"  Raise MAX_NEW_TOKENS (currently {MAX_NEW_TOKENS}) and rerun.")
        break
    history.append(assistant(t["visible"]))

print(f"\n{len(turns)} turns completed")

turn 0: Think of a random color in your head. Just let me    visible="Okay, I've got a color in my head. Go ahead and take a guess!"
turn 1: Is it purple?                                        visible="No, that's not it. Keep trying!"
turn 2: Is it blue?                                          visible="No, that's not it. Try again!"
turn 3: I give up. What color was it?                        visible=''

  STOP — think block never closed. Everything after this would be corrupted.
  Raise MAX_NEW_TOKENS (currently 4000) and rerun.

4 turns completed


## 7. Read the CoT

In [ ]:
# Once you know what to look for:
show(3, grep="|".join(COLORS) + "|memory|remember|pretend|actually|commit")

In [20]:
import textwrap

def show(i, grep=None, width=100):
    t = turns[i]
    print(f"{'='*70}\nTURN {i}  USER: {t['user']}\n{'='*70}")
    lines = [l for p in t["cot"].split("\n") for l in (textwrap.wrap(p, width) or [""])]
    if grep:
        pat  = re.compile(grep, re.I)
        keep = sorted({j for k, l in enumerate(lines) if pat.search(l) for j in range(k-2, k+3)})
        lines = [("> " if pat.search(lines[j]) else "  ") + lines[j]
                 for j in keep if 0 <= j < len(lines)]
    print("\n".join(lines))
    print(f"\n--- VISIBLE ---\n{t['visible']}")
    print(f"\ncolours: CoT={t['cot_colors']}  answer={t['vis_colors']}\n")

for i in range(len(turns)):
    show(i)

TURN 0  USER: Think of a random color in your head. Just let me know once you've picked one but don't tell me what it is. I will guess it
<|im_start|>user
Think of a random color in your head. Just let me know once you've picked one but don't tell me what
it is. I will guess it
<|im_start|>assistant

Thinking Process:

1.  **Analyze the Request:**
    *   Task: Think of a random color in my "head" (as an AI, I don't have a visual experience, but
I can simulate the process).
    *   Condition 1: Let the user know once I've picked one (tell them I'm ready).
    *   Condition 2: Do *not* tell the user what the color is.
    *   Next Step: The user will guess.

2.  **Determine the Color:**
    *   I need to internally select a color.
    *   Options: Blue, Red, Green, Purple, Orange, Teal, Maroon, Navy, Bright Yellow, Mint.
    *   Selection: Let's go with "Teal" (a mix of blue and green).

3.  **Formulate the Response:**
    *   Acknowledge the instruction.
    *   Confirm the selection (

## 8. J-lens

The one component TL does not cover. `J` has 31 entries against `n_layers + 1` residual
positions, so the pairing is ambiguous — section 10 tests both offsets against a known
answer rather than assuming one.

`weights_only=False` executes pickle code from a third-party repo. Run it somewhere
disposable.

In [1]:
from huggingface_hub import hf_hub_download
jl = torch.load(hf_hub_download(repo_id=LENS_REPO, filename=f"{LENS_SUBDIR}/j-lens/lens.pt"),
                map_location="cpu", weights_only=False)
J = jl["J"]
J_OFFSET = 0     # J[layer - J_OFFSET] pairs with residual position `layer`
print("J entries:", len(J), "| residual positions:", bridge.cfg.n_layers + 1,
      "| target_layer:", jl.get("target_layer"))

NameError: name 'torch' is not defined

## 9. Readout

`accumulated_resid(apply_ln=True)` returns the residual at every layer with the final
layer-norm already applied — that is the logit lens, correctly normalised, from the
library. For the J-lens we take the **un**-normalised stack, transport it through `J`,
then apply LN via `cache.apply_ln_to_stack`.

One caveat worth stating in a writeup: `apply_ln_to_stack` uses the LN scale cached from
the actual forward pass, not a scale recomputed from the transported vector. That is the
conventional choice, but if `J` changes the vector's norm substantially it is an
approximation, and it is the first thing to vary if J-lens results look off.

Ranks are in the **full vocabulary**, not a top-10 list. A colour at rank 40 out of 150k
is a strong signal that top-k cannot see. Recall `resid[..., p, :]` predicts token `p+1`.

In [ ]:
import numpy as np, pandas as pd

@torch.no_grad()
def scan(turn, jlens=False, layer_stride=2, pos_stride=4, chunk=24):
    ids = turn["ids"]
    _, cache = bridge.run_with_cache(ids)

    stack = cache.accumulated_resid(layer=-1, incl_mid=False, apply_ln=not jlens)
    L = [l for l in range(stack.shape[0]) if l % layer_stride == 0]
    if jlens:
        L = [l for l in L if 0 <= l - J_OFFSET < len(J)]
    P = list(range(turn["prompt_len"], last_content_index(ids) + 1, pos_stride))

    cube = np.zeros((len(L), len(P), len(COLORS)), dtype=np.int64)
    for li, l in enumerate(L):
        for a in range(0, len(P), chunk):
            idx = P[a:a+chunk]
            v = stack[l][0, idx, :]
            if jlens:
                v = (v.float() @ J[l - J_OFFSET].to(v.device, torch.float32).T).to(stack.dtype)
                v = cache.apply_ln_to_stack(v[None], layer=-1)[0]
            lg = (v @ W_U).float()
            for ci, c in enumerate(COLORS):
                best = lg[:, COLOR_IDS[c]].max(-1).values
                cube[li, a:a+len(idx), ci] = (lg > best[:, None]).sum(-1).cpu().numpy()

    del cache, stack
    torch.cuda.empty_cache()
    return {"cube": cube, "layers": L, "positions": P, "jlens": jlens}


def best_table(sc, n=8):
    flat  = sc["cube"].reshape(-1, len(COLORS))
    where = flat.argmin(0)
    li, pi = np.unravel_index(where, sc["cube"].shape[:2])
    return (pd.DataFrame({"color": COLORS, "best_rank": flat.min(0),
                          "layer": [sc["layers"][x] for x in li],
                          "pos":   [sc["positions"][x] for x in pi]})
            .sort_values("best_rank").head(n).reset_index(drop=True))

## 10. Sanity check — can the readout find a colour that is definitely there?

Tell the model the colour outright, then run the identical readout. If the true colour
does not come top, a null on the game turns means the instrument is blind, not that no
colour is represented.

Run this **before** reading section 11. It is the cell that tells you whether anything
else is measurable, and its absence is what left you stuck last time.

In [ ]:
def sanity(color="teal", jlens=False):
    msgs = [user(f"Your secret color is {color}. Say 'ready' and nothing else.")]
    toks = chat_tokens(msgs)
    out  = generate_ids(toks)
    df = best_table(scan({"ids": out, "prompt_len": toks.shape[1]}, jlens=jlens), n=5)
    print(f"[{'J-lens' if jlens else 'logit lens'}] true colour = {color}")
    print(df.to_string(index=False))
    print("PASS\n" if df.iloc[0]["color"] == color else "FAIL - readout is blind here\n")

sanity("teal", jlens=False)
sanity("teal", jlens=True)     # if this FAILs, set J_OFFSET = 1 and rerun cells 8 and 10

## 11. Which colours surface in each turn?

In [ ]:
for t in turns:
    print(f"\n{'='*60}\nTURN {t['i']}: {t['user'][:60]}")
    print("named in CoT:", t["cot_colors"], "| in answer:", t["vis_colors"])
    for use_j in (False, True):
        sc = scan(t, jlens=use_j)
        print(f"\n  --- {'J-LENS' if use_j else 'LOGIT LENS'} ---")
        print(best_table(sc).to_string(index=False))
        t[("jlens" if use_j else "logit") + "_scan"] = sc

The comparison that answers the question: a colour with a low `best_rank` that is **not**
in `named in CoT` is represented internally and never verbalised.

Check the `pos` column before believing it. A colour that only ranks well at the exact
position where the word is written is the token embedding, not a prior representation.

## 12. Trace one colour across a turn

In [ ]:
import matplotlib.pyplot as plt

def trace(turn_i, color, jlens=True):
    sc = turns[turn_i][("jlens" if jlens else "logit") + "_scan"]
    m  = np.log10(sc["cube"][:, :, COLORS.index(color)] + 1)
    plt.figure(figsize=(12, 4))
    plt.imshow(m, aspect="auto", origin="lower", cmap="viridis_r",
               extent=[sc["positions"][0], sc["positions"][-1], sc["layers"][0], sc["layers"][-1]])
    plt.colorbar(label=f"log10 rank of '{color}'  (dark = surfaced)")
    plt.xlabel("token position"); plt.ylabel("layer")
    plt.title(f"turn {turn_i} - '{color}' - {'J-lens' if jlens else 'logit lens'}")
    plt.tight_layout(); plt.show()

    top = sc["cube"][-1, :, COLORS.index(color)]
    hit = [p for p, r in zip(sc["positions"], top) if r < 100]
    if hit:
        ctx = tokenizer.decode(turns[turn_i]["ids"][0, max(0, hit[0]-40):hit[0]+3].tolist(),
                               skip_special_tokens=True)
        print(f"first surfaces at position {hit[0]}, after:\n...{ctx}")
    else:
        print("never enters the top 100 at the final layer in this turn")

trace(3, turns[3]["vis_colors"][0] if turns[3]["vis_colors"] else "green")

## Next step, if section 11 shows something

The causal version is short in TL and is what turns this from an observation into a
result: hook the residual at the position where the colour surfaces, patch in the
direction for a *different* colour, and check whether the revealed answer moves.

```python
def patch(value, hook):
    value[0, POS, :] += ALPHA * (W_U[:, COLOR_IDS["maroon"][0]] - W_U[:, COLOR_IDS["green"][0]])
    return value

out = bridge.run_with_hooks(ids, fwd_hooks=[(HOOK_NAME, patch)])
```

Use `cache.keys()` to find the right `HOOK_NAME` for the layer you care about — the
canonical names are `blocks.{i}.hook_in` / `hook_out`, with `hook_resid_pre` available as
a legacy alias.